# Validation Test - Two-phase Couette Flow

This notebook and the corresponding evaluation notebook (TwoPhaseCouetteFlow_Evaluation.ipynb) are part of published results (cf. 5.3) found in *The extended discontinuous Galerkin method adapted for moving contact line problems via the generalized Navier boundary condition* (https://doi.org/10.1002/fld.5016).

In [ ]:
#r "BoSSSpad.dll"
//#r "..\..\..\src\L4-application\BoSSSpad\bin\Release\net8.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
ExecutionQueues

In [ ]:
string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
userName

In [ ]:
//var myBatch = ExecutionQueues[1];
var myBatch = GetDefaultQueue();
myBatch

In [ ]:
wmg.Init("CLpaper_TwoPhaseCouetteFlow", myBatch);

In [ ]:
wmg.SetNameBasedSessionJobControlCorrelation();

In [ ]:
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

## Sessions

In [ ]:
static var sessions = BoSSSshell.WorkflowMgm.Sessions;
sessions

In [ ]:
// foreach (var s in sessions) {
//     if(s.Name.Contains("CouetteGNBC") && s.CreationTime.Date == new DateTime(2025, 11, 25)) {
//         s.Delete(true);
//         //Console.WriteLine($"Deleted session from {s.CreationTime.Date}");
//     }
// }

In [ ]:
// static var sessions = BoSSSshell.WorkflowMgm.Sessions;
// sessions

## Grid Creation for Mesh Study

In [ ]:
int[] Resolutions = new int[] { 10, 12, 16 };
IGridInfo[] Grids = new IGridInfo[Resolutions.Length];

double L = 27.2;
double H = 13.6;

for(int i = 0; i < Resolutions.Length; i++) {
    int Res = Resolutions[i];
    string GridName = $"TwoPhaseCouetteFlow_{Res}";

    IGridInfo cachedGrid = wmg.Grids.FirstOrDefault(grid => grid.Name == GridName);
    cachedGrid = null;
    if(cachedGrid == null) {
        
        // must create new Grid
        double[] Xnodes = GenericBlas.Linspace(0, 4 * L, 8 * Res + 0);
        double[] Ynodes = GenericBlas.Linspace(0, H, Res + 1);
        var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes, periodicX: true);
    
        grd.Name = GridName;
    
        grd.EdgeTagNames.Add(1, "navierslip_linear_lower");
        grd.EdgeTagNames.Add(2, "navierslip_linear_upper");

        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 0;
            if (Math.Abs(X[1]) <= 1.0e-8)
                et = 1;
            if (Math.Abs(X[1] - H) <= 1.0e-8)
                et = 2;
            return et;
        });
        
        Grids[i] = wmg.SaveGrid(grd);
        
    } else {
        //Console.WriteLine($"type: {cachedGrid.GetType()}, is IGridInfo? {cachedGrid is IGridInfo}");
        Console.WriteLine("Grid already found in database - identifid by name " + GridName);
        Grids[i] = cachedGrid;
    }
    
}

## Initial Values & Boundary Values

In [ ]:
var PhiFunc = new Formula(
    "Phi",
    false,
    "using ilPSP.Utils; " + 
    "double Phi(double[] X) { " + 
    "    return (Math.Abs(X[0] - 2 * 27.2) - 27.2); " + 
    "}");

In [ ]:
var VelX_lowerWall_symmetric = new Formula(
    "VelXlowerWallsymmetric",
    false,
    "using ilPSP.Utils; " + 
    "double VelXlowerWallsymmetric(double[] X) { " + 
    "    return -0.25; " + 
    "}");

In [ ]:
var VelX_upperWall_symmetric = new Formula(
    "VelXupperWallsymmetric",
    false,
    "using ilPSP.Utils; " + 
    "double VelXupperWallsymmetric(double[] X) { " + 
    "    return 0.25; " + 
    "}");

In [ ]:
var VelX_lowerWall_asymmetric = new Formula(
    "VelXlowerWallasymmetric",
    false,
    "using ilPSP.Utils; " + 
    "double VelXlowerWallasymmetric(double[] X) { " + 
    "    return -0.2; " + 
    "}");

In [ ]:
var VelX_upperWall_asymmetric = new Formula(
    "VelXupperWallasymmetric",
    false,
    "using ilPSP.Utils; " + 
    "double VelXupperWallasymmetric(double[] X) { " + 
    "    return 0.2; " + 
    "}");

## Physical Settings

In [ ]:
double rho = 0.81;    
double mu = 1.95;
double sigma = 5.5;

In [ ]:
double dt = 0.02;
static double t_end = 160;

## Control settings

In [ ]:
string[] caseS = new string[] { "symmetric", "asymmetric" };

In [ ]:
int[] degS = new int[] { 4, 3, 2 };

In [ ]:
List<XNSE_Control> Controls = new List<XNSE_Control>();

In [ ]:
foreach (string test in caseS) {
for(int iStudy = 0; iStudy < degS.Length; iStudy++) {
    
    XNSE_Control C = new XNSE_Control();
    
    int pDeg = degS[iStudy];   
    var grd  = Grids[iStudy];

    C.SetDGdegree(pDeg);
    
    // set grid
    C.SetGrid(grd);
    
    // initial conditions   
    C.AddInitialValue("Phi", PhiFunc);   

    // boundary conditions
    if (test == "symmetric") {  
        C.AddBoundaryValue("navierslip_linear_lower", "VelocityX#A", VelX_lowerWall_symmetric);
        C.AddBoundaryValue("navierslip_linear_lower", "VelocityX#B", VelX_lowerWall_symmetric);
        C.AddBoundaryValue("navierslip_linear_upper", "VelocityX#A", VelX_upperWall_symmetric);
        C.AddBoundaryValue("navierslip_linear_upper", "VelocityX#B", VelX_upperWall_symmetric);
    } else if (test == "asymmetric") {
        C.AddBoundaryValue("navierslip_linear_lower", "VelocityX#A", VelX_lowerWall_asymmetric);
        C.AddBoundaryValue("navierslip_linear_lower", "VelocityX#B", VelX_lowerWall_asymmetric);
        C.AddBoundaryValue("navierslip_linear_upper", "VelocityX#A", VelX_upperWall_asymmetric);
        C.AddBoundaryValue("navierslip_linear_upper", "VelocityX#B", VelX_upperWall_asymmetric);
    }
    
    C.PostprocessingModules.Add(new MovingContactLineLogging(){ SolverStage = 2, LogPeriod = 5});

    C.PhysicalParameters.rho_A = rho;
    C.PhysicalParameters.rho_B = rho;
    C.PhysicalParameters.mu_A  = mu;
    C.PhysicalParameters.mu_B  = mu;
    C.PhysicalParameters.Sigma = sigma;

    C.PhysicalParameters.IncludeConvection = true;
    C.PhysicalParameters.Material = true;

    C.AdvancedDiscretizationOptions.SST_isotropicMode = SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;
    C.AdvancedDiscretizationOptions.GNBC_SlipLength = NavierSlip_SlipLength.Prescribed_Beta;
    C.PhysicalParameters.betaL = 0.0;
    if (test == "symmetric") {
        C.PhysicalParameters.betaS_A = 1.5;
        C.PhysicalParameters.betaS_B = 1.5;
        C.PhysicalParameters.theta_e = Math.PI / 2.0;
    }
    else if (test == "asymmetric") {
        C.PhysicalParameters.betaS_A = 0.591;
        C.PhysicalParameters.betaS_B = 1.5;
        C.PhysicalParameters.theta_e = Math.Acos(0.38);
    }
    
    C.LSContiProjectionMethod = ContinuityProjectionOption.ConstrainedDG;

    C.NonLinearSolver.MaxSolverIterations = 50;
    
    C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
    C.Timestepper_LevelSetHandling = LevelSetHandling.Coupled_Once;
    C.Option_LevelSetEvolution = LevelSetEvolution.FastMarching;
    C.TimeSteppingScheme = TimeSteppingScheme.BDF3;
  
    C.dtMin         = dt;
    C.dtMax         = dt;
    C.Endtime       = t_end;
    C.NoOfTimesteps = (int)(t_end/dt); 
    
    C.saveperiod = 50;
    
    C.SessionName = $"CouetteGNBC_{test}_compDOFstudy_k{degS[iStudy]}_mesh{iStudy}";
    
    Controls.Add(C);
}
}

In [ ]:
foreach (var ctrl in Controls) {
    Console.WriteLine($"{ctrl.SessionName}");
    // Console.WriteLine($"{ctrl.SessionName}: dt = {ctrl.dtMin}; saveperiod = {ctrl.saveperiod}; logperiod = {ctrl.PostprocessingModules.Pick(0).LogPeriod}");
}

In [ ]:
int NoCtrls = Controls.Count();
NoCtrls

## Launch/Restart Jobs

In [ ]:
public static Job CreateRestartJob(Job job2rest) {

    string baseName = job2rest.LatestSession.Name;

    // find latest session with same base name (in case of multiple restarts)
    var studySess = sessions.Where(sess => sess.Name.Contains(baseName));
    int studyCount = studySess.Count();
    studySess = studySess.OrderBy(sess => sess.CreationTime);
    var LatestSession = studySess.Last();
    if (LatestSession.SuccessfulTermination) {
        Console.WriteLine($"Latest session {LatestSession.Name} already terminated successfully; no restart job created.");
        return LatestSession.GetControl().CreateJob();
    }
    Console.WriteLine($"Creating restart job for session {LatestSession.Name} ...");

    var rCtrl = LatestSession.GetControl().CloneAs();
    // var rCtrl = job2rest.LatestSession.GetControl().CloneAs();

    Guid rst_ID = LatestSession.ID;
    TimestepNumber rst_ts = LatestSession.Timesteps.Last().TimeStepNumber;

    rCtrl.InitialValues.Clear();
    rCtrl.InitialValues_Evaluators.Clear();

    //rCtrl.Endtime = t_end;
    rCtrl.RestartInfo = Tuple.Create(rst_ID, rst_ts);

    rCtrl.SessionName = baseName + "_Restart" + studyCount;

    var rJob = rCtrl.CreateJob();

    // set resources
    rJob.NumberOfMPIProcs = job2rest.NumberOfMPIProcs;

    int numThreads = job2rest.NumberOfThreads;
    rJob.NumberOfThreads = numThreads;
    //IDictionary<string, string> args = rJob.EnvironmentVars;
    //args["BOSSS_ARG_7"] = numThreads.ToString();
    //args.Remove("OMP_NUM_THREADS");
    //rJob.MySetCommandLineArguments(args.Values.ToArray());

    rJob.RetryCount = job2rest.RetryCount;

    Console.WriteLine($"Restart job for session {rCtrl.SessionName} created.");

    return rJob;
}

In [ ]:
List<Job> jobs = new List<Job>();

In [ ]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();
    
    oneJob.NumberOfMPIProcs = 1;

    //System.Environment.SetEnvironmentVariable("OMP_NUM_THREADS", "2");
    int numThreads = 1;
    oneJob.NumberOfThreads = numThreads;
    //IDictionary<string, string> args = oneJob.EnvironmentVars;
    //args["BOSSS_ARG_7"] = numThreads.ToString();
    //args.Remove("OMP_NUM_THREADS");
    //oneJob.MySetCommandLineArguments(args.Values.ToArray());

    //oneJob.UseComputeNodesExclusive = true;\n",

    //((SlurmClient)myBatch).ExecutionTime = "24:00:00";

    oneJob.RetryCount = 1;
    oneJob.Activate(myBatch);
    
    if (oneJob.Status == JobStatus.FailedOrCanceled) {
        Console.WriteLine($"Job for session {ctrl.SessionName} failed. Trying to restart latest job ...");

        var restartJob = CreateRestartJob(oneJob);
        restartJob.Activate(myBatch);
        jobs.Add(restartJob);

    } else {   
          
        //oneJob.Activate(myBatch);
        jobs.Add(oneJob);
    }
}

## Wait for Completion and Check Job Status

In [ ]:
int NoInProgress = jobs.Where(job => job.Status == JobStatus.InProgress 
                                || job.Status == JobStatus.PendingInExecutionQueue
                                || job.Status == JobStatus.PreActivation).Count();
NoInProgress

In [ ]:
int maxDays = 7;
int waitingDays = 0;
while (NoInProgress > 0 && waitingDays < maxDays) {
    wmg.BlockUntilAllJobsTerminate(3600*24*1); // wait one day for the jobs to finish
    NoInProgress = jobs.Where(job => job.Status == JobStatus.InProgress).Count();
    waitingDays++;
}

In [ ]:
int NoFailed = jobs.Where(job => job.Status == JobStatus.FailedOrCanceled).Count();
NoFailed

In [ ]:
int NoSuccess = jobs.Where(job => job.Status == JobStatus.FinishedSuccessful).Count();
NoSuccess

In [ ]:
// check for restart sessions
if (NoFailed > 0) {
    foreach (var ctrl in Controls) {
        var job = ctrl.GetJob();
        if (job.Status == JobStatus.FailedOrCanceled) {
            var studySess = sessions.Where(sess => sess.Name.Contains(ctrl.SessionName));
            int studyCount = studySess.Count();

            if (studyCount > 1) {
                Console.WriteLine("Restart session found! Take last run");       
                bool success = studySess.Where(sess => sess.Name.EndsWith($"_restart{studyCount-1}")).Single().SuccessfulTermination;
                if (success) {
                    Console.WriteLine($"Restart session {ctrl.SessionName}_restart{studyCount-1} was successful.");
                    NoFailed--;
                    NoSuccess++;
                } else {
                    Console.WriteLine($"Restart session {ctrl.SessionName}_restart{studyCount-1} also failed.");
                }
            } else if (studyCount == 1) {
                Console.WriteLine("No restart session found. {ctrl.SessionName} might to be restarted.");
            } else { // studyCount == 0
                throw new ApplicationException($"No session found for {ctrl.SessionName}!"); // should not happen
            } 
        }
    }

}

In [ ]:
NUnit.Framework.Assert.AreEqual(0, NoFailed, $"Some simulations failed.");

In [ ]:
NUnit.Framework.Assert.AreEqual(NoCtrls, NoSuccess, $"Not all simulations finished successfully.");